# Working with structured data - Part 3

This notebook accompanies the script <strong><span style="color:red;">07_pandas_part_C.pdf</span></strong>  and provides practical examples related to its content.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

<hr style="border: none; height: 20px; background-color: green;">

# 9. Exploring Datasets with GroupBy and Aggregation

<hr style="border: none; height: 10px; background-color: LightBlue;">

### Load and explore the Seaborn planets dataset

In [ ]:
planets = sns.load_dataset("planets")

In [ ]:
# First rows as a quick data preview
planets.head()

In [ ]:
# This dataset features multiple numerical columns with missing values
planets.info()

In [ ]:
# Example: Missing values per column, sorted, percent, 1 decimal place
(planets.isna()
        .mean()
        .mul(100)
        .round(1)
        .sort_values(ascending=False)
        .to_frame('missing_rate') 
        )

#### Verifying the 2010+ Kepler discovery claim

In [ ]:
before = planets["year"] < 2010
after = ~before 

print(f"Before 2010: {before.sum()} planets")
print(f"2010 or after: {after.sum()} planets")

In [ ]:
# detections per year
planets['year'].value_counts().sort_index()

In [ ]:
# Histogram of discovery years to visualize distribution
planets['year'].plot(kind='hist', bins=26)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Basic Example: Split–Apply–Combine in code

- Calculate median planet mass for each discovery method.  
- Sorted from highest to lowest

In [ ]:
# Split
group = planets.groupby("method")

# Apply + Combine
result = group["mass"].count()

# Optional post-processing
result.sort_values(ascending=False)

In [ ]:
# Same as a single command
planets.groupby("method")['mass'].count().sort_values(ascending=False)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Aggregating one or many keys, columns, funktions

#### One key one column one function

In [ ]:
# SeriesGroupBy
planets.groupby("method")["mass"].sum()

In [ ]:
# DataFrameGroupBy
planets.groupby("method")[["mass"]].sum()

#### One key two columns one function

In [ ]:
planets.groupby("method")[["mass", "distance"]].sum()

#### Two key one columns one function

In [ ]:
planets.groupby(["method", "year"])[["mass"]].sum()

#### One key one columns two functions

In [ ]:
planets.groupby(["method", "year"]).agg({"distance": ["min", "max"]})

#### One key multiple columns and functions

In [ ]:
(
    planets.groupby("method")
    .agg(
        rows=("method", "size"),
        mass_cnt=("mass", "count"), 
        mass_median=("mass", "median"),
        distance_median=("distance", "median"),
        year_min=("year", "min"),
        year_max=("year", "max"),
    )
    .sort_values("rows", ascending=False)
)


<hr style="border: none; height: 5px; background-color: gray;">

### Interpreting `size()` vs. `count()`

- If `size()` is much larger than `count()`, many values are missing in that column.
- Use both metrics together to understand how much valid data each group contains.


In [ ]:
df = planets.groupby("method").agg(
    size=("method", "size"),
    count=("mass", "count"),
    median=("mass", "median"),
)
df.sort_values("size", ascending=False)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## transform

Returns the same length as the input 

In [ ]:
# Fills each row with the mean mass for its method group
df_result = planets.groupby("method")[["mass"]].transform("median")

print(f"Row counts {planets.shape[0]} == {df_result.shape[0]}")

df_result.head()

In [ ]:
# Fills each row with the mean mass for its (method, year) group
df_result = planets.groupby(["method", "year"])[["mass"]].transform("median")

print(f"Row counts {planets.shape[0]} == {df_result.shape[0]}")

df_result.head()

In [ ]:
# Impute missing values using group statistics
planets["mass_filled"] = planets["mass"].fillna(
    planets.groupby("method")["mass"].transform("median")
)
planets[["method", "mass", "mass_filled"]].iloc[5:10]

#### Example: Z‑Scoring

Z‑scoring standardizes a numerical value so that it reflects **how many standard deviations** it is **above or below the mean** of its comparison group.   
This transformation removes the influence of differing scales or units and **allows direct comparison across groups**.

In [ ]:
mass_mean = planets.groupby("method")["mass_filled"].transform("mean")
mass_std = planets.groupby("method")["mass_filled"].transform("std")

planets["mass_z"] = (planets["mass_filled"] - mass_mean) / mass_std

planets[["method", "mass", "mass_filled", "mass_z"]].head(10)

<hr style="border: none; height: 5px; background-color: gray;">

#### `transform` cannot combine multiple columns within a group


Attempting to compute a new column based on multiple existing columns within `transform` fails, because the function only receives a single column (`Series`) at a time.  
Not the entire group `DataFrame`.

In [ ]:
def mass_distance_ratio(group):
    # Tries to access two columns. NOT possible with transform!
    group['mass_per_distance'] = group['mass'] / group['distance']
    return group

try:
    planets['mass_per_distance'] = planets.groupby('method').transform(mass_distance_ratio)
except KeyError as e:
    print(f"KeyError: {e}")

In [ ]:
# use apply to access multiple columns
planets.groupby('method').apply(mass_distance_ratio).dropna()

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Custom functions: `apply()` when needed


In [ ]:
def mass_range(group):
    return group["mass"].max() - group["mass"].min()

planets.groupby("method").apply(mass_range)

In [ ]:
def summarize_group(group: pd.DataFrame) -> pd.Series:
    mass = group["mass"]
    return pd.Series(
        {
            "n_rows": len(group),
            "mass_non_missing": mass.count(),
            "mass_missing_rate": mass.isna().mean().round(3),
            "mass_median": mass.median(),
            "distance_median": group["distance"].median(),
        }
    )

custom_summary = (
    planets.groupby("method", group_keys=False)
    .apply(summarize_group)
    .sort_values(by="n_rows", ascending=False)
)

custom_summary

<hr style="border: none; height: 10px; background-color: LightBlue;">

## `filter()`

In [ ]:
def at_least_50_discoveries(group):
    return len(group) >= 50

filtered = planets.groupby("year").filter(at_least_50_discoveries)

dropped = planets.shape[0] - filtered.shape[0]
print(f"Dropped {dropped} rows from years with fewer than 50 discoveries")

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Tips dataset

In [ ]:
tips = sns.load_dataset("tips")
tips.head(8)

In [ ]:
tips.head()

In [ ]:
pd.crosstab(
    index=tips["time"],
    columns=tips["day"],
)

In [ ]:
tips['size'].value_counts().sort_index()

<hr style="border: none; height: 10px; background-color: LightBlue;">

## `groupby() + unstack()`

Use `groupby()` with `unstack()` to aggregate data and reshape the result into a wide, table-like format

In [ ]:
tips.groupby(["day", "time"])[["tip"]].mean()

In [ ]:
tips.groupby(["day", "time"])[["tip"]].mean().unstack()

In [ ]:
tips.groupby(["day", "time"])[["tip"]].sum().unstack()

In [ ]:
tips.groupby(["day", "time", "sex"])[["tip"]].sum().unstack()

In [ ]:
tips.groupby(['day', 'time'])[['total_bill','tip']].mean().unstack()

In [ ]:
tips.groupby(['day', 'time', 'sex'])[['total_bill','tip']].mean().unstack()

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Pivot tables

Use `pivot()` to rearrange data by turning unique values from one column into new columns.



In [ ]:
pd.pivot_table(
    tips,
    index='day',
    columns='time',
    values='tip',
)

In [ ]:
# pivot_table() is also available as a DataFrame method
tips.pivot_table(
    index='day',
    columns='time',
    values='tip',
)

In [ ]:
pd.pivot_table(
    tips,
    index='day',
    columns='sex',
    values='tip',
    margins=True,
    aggfunc=['count', 'mean'],
)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## `crosstab()`

Use `crosstab()` to compute a cross-tabulation of two or more factors, often for counting or aggregation.

In [ ]:
pd.crosstab(
    index=tips["day"],
    columns=tips["sex"],
)

In [ ]:
# Extreme example with multiple crosstab options
pd.crosstab(
    index=[tips["day"], tips["smoker"]],
    columns=[tips["sex"], tips["time"]],
    values=tips["tip"],
    aggfunc="mean",             # can use only one aggfunc when normalize is set
    margins=True,               # adds totals row/column
    margins_name="Total",       # name for the totals row/column
    normalize="all",            # show as proportion of grand total
    dropna=True,                # drop rows with NA
)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## `pd.cut()`

Use `pd.cut()` to segment and sort numeric data into discrete intervals or bins.



In [ ]:
tips['bill_bins'] = pd.cut(tips['total_bill'], 
                           bins=[0, 10, 20, 50], 
                           labels=['Low', 'Medium', 'High'])

tips[['total_bill', 'bill_bins']].head()

In [ ]:
# Same but with open-ended below and above bins
tips['bill_bins'] = pd.cut(tips['total_bill'], 
                           bins=[-np.inf, 10, 20, np.inf],   # open-ended below and above
                           labels=['Low', 'Medium', 'High'])

tips[['total_bill', 'bill_bins']].head()

In [ ]:
# Automatically create 3 equal-width bins over the range of 'total_bill'
tips['bill_bins'] = pd.cut(tips['total_bill'], 
                           bins=3,
                           labels=['Low', 'Medium', 'High'])
tips[['total_bill', 'bill_bins']].head()

<hr style="border: none; height: 10px; background-color: LightBlue;">

## More GroupBy Examples with the Titanic Dataset

In [ ]:
titanic = sns.load_dataset("titanic")
titanic.info()

In [ ]:
titanic.head()

#### Survival Rate by Sex and Class

In [ ]:
# Calculate the mean survival rate for each combination of sex and ticket class
titanic.groupby(['sex', 'class'])['survived'].mean().unstack()

#### Median Age of Survivors by Class

In [ ]:
# For each ticket class, compute the median age among survived passengers
titanic[titanic['survived'] == 1].groupby('class')[['age']].median()

#### Survival Rate by Age Group

In [ ]:
# Bin ages into categories and calculate survival rate per age group
age_bins = pd.cut(titanic['age'], [0, 12, 18, 60, 80], labels=['child', 'teen', 'adult', 'senior'])
titanic.groupby(age_bins)['survived'].mean()

#### Survival Rates: First Class Advantage by Sex

In [ ]:
# Compare survival rate for first class vs. non-first class, broken down by sex
df_unstacked = titanic.groupby(['sex', titanic['class'] == 'First'])['survived'].mean().unstack()
df_unstacked.columns = ['not_first_class', 'first_class']  # rename columns for clarity
df_unstacked

#### Survival Rate for the Three Most Common Surnames

In [ ]:
# Find the three most frequent surnames and compute their survival rates
titanic['surname'] = titanic['who'].str.split().str[-1]
top3 = titanic['surname'].value_counts().head(3).index
titanic[titanic['surname'].isin(top3)].groupby('surname')['survived'].mean()

<hr style="border: none; height: 20px; background-color: green;">

# 10. Vectorized string operations

In [ ]:
# Series of standard Python strings (dtype 'object')
monty_py_str = pd.Series([
    'Graham Chapman', 'John Cleese', 'Terry Gilliam',
    'Eric Idle', 'Terry Jones', 'Michael Palin', None
])
monty_py_str

In [ ]:
# Series with pandas StringDtype (better for text operations and missing values)

monty_pd_string = pd.Series([
    'Graham Chapman', 'John Cleese', 'Terry Gilliam',
    'Eric Idle', 'Terry Jones', 'Michael Palin', None
], dtype=pd.StringDtype())
monty_pd_string

<hr style="border: none; height: 5px; background-color: gray;">

### Some Examples

In [ ]:
monty = pd.Series([
    'Graham Chapman', 'John Cleese', 'Terry Gilliam',
    'Eric Idle', 'Terry Jones', 'Michael Palin'
], dtype=pd.StringDtype())

In [ ]:
# Convert all names to lowercase
monty.str.lower()

In [ ]:
# Check if each name starts with 'T'
monty.str.startswith('T')

In [ ]:
monty[monty.str.startswith('T')] 

In [ ]:
# Split each name into a list of words
monty.str.split() # default: split on whitespace

#### Note: `.str` also works on indexable results

- `.str` is not limited to plain strings
- It can also access elements of other indexable objects produced along the way
- But many `.str` methods still expect actual strings

In [ ]:
# Get the first word (usually the first name) from each entry
monty.str.split().str[0]

In [ ]:
# Replace spaces with underscores (literal replacement)
print(monty.str.replace(' ', '_', regex=False))

In [ ]:
# Replace spaces with underscores (regex)
monty.str.replace(r'\s+', '_', regex=True)

In [ ]:
# Extract the first sequence of letters (e.g. first name) from each string
monty.str.extract('([A-Za-z]+)')

In [ ]:
# Extracts the whole name if it starts with a consonant and ends with a consonant
monty.str.extract(r'^([^AEIOU].*[^aeiou])$')

In [ ]:
# Slice characters from position 0 to 6
monty.str.slice(0, 6)

In [ ]:
# Replace characters between positions 0 and 6 with 'NAME'
monty.str.slice_replace(0, 6, 'NAME')

In [ ]:
# Concatenate all strings with a separator
monty.str.cat(sep=', ')

In [ ]:
# Repeat each string twice
monty.str.repeat(2)

In [ ]:
# Join characters in each string with a separator
monty.str.join('-')

In [ ]:
# Create dummy variables from separated tokens
monty.str.get_dummies(sep=' ')

In [ ]:
full_monte = pd.DataFrame({
    'name': monty,
    'info': ['CBD', 'BD', 'CA', 'BD', 'CB', 'BCD']

})
full_monte

In [ ]:
full_monte["info"].str.get_dummies(sep="")

<hr style="border: none; height: 20px; background-color: green;">

# 11. Working with time series data

<hr style="border: none; height: 10px; background-color: LightBlue;">

## **Python** core time modules

- `datetime`: creating, manipulating, comparing, and formatting dates/times
- `calendar`: calendar-based info (weekday names, leap years)
- `time`: system time, UNIX timestamps, delays
- `dateutil`: flexible string parsing (not standard lib, but widely used)
   
#### Import modules

In [ ]:
import calendar
import time
from datetime import datetime, timedelta, UTC
from zoneinfo import ZoneInfo

from dateutil import parser

<hr style="border: none; height: 5px; background-color: gray;">

### Datetime modul basics

In [ ]:
# Current timestamp from the OS (local system time), without timezone
now_no_timezone = datetime.now()
print("type:",type(now_no_timezone))
print(now_no_timezone)

In [ ]:
# No timezone set → tzinfo and utcoffset() are None
print("tzinfo:   ",now_no_timezone.tzinfo)
print("utcoffset:",now_no_timezone.utcoffset())

In [ ]:
# Set timezone 
now = datetime.now(ZoneInfo("Europe/Zurich"))
print(now)

In [ ]:
print("---------------------------------------------")
print("type:       ",type(now))
print("---------------------------------------------")
print("year:       ", now.year)
print("month:      ", now.month)
print("day:        ", now.day)
print("hour:       ", now.hour)
print("minute:     ", now.minute)
print("second:     ", now.second)
print("microsecond:", now.microsecond)
print("---------------------------------------------")
print("weekday:    ", now.weekday())
print("isoformat:  ", now.isoformat())
print("tzinfo:     ", now.tzinfo)
print("utcoffset:  ", now.utcoffset())
print("timestamp:  ", now.timestamp())
print("---------------------------------------------")

In [ ]:
# timedelta example
time_delta = datetime.now(ZoneInfo("Europe/Zurich")) - now

print("---------------------------------------------")
print("type:          ",type(time_delta))
print("---------------------------------------------")
print("days:          ", time_delta.days)
print("seconds:       ", time_delta.seconds)
print("microseconds:  ", time_delta.microseconds)
print("total_seconds: ", time_delta.total_seconds())
print("---------------------------------------------")
print("resolution:    ", time_delta.resolution)
print("---------------------------------------------")

In [ ]:
# Add days with timedelta
future = now + timedelta(days=7)
print(future)

In [ ]:
# Compare datetimes
datetime(2025, 3, 27) > datetime(2025, 3, 20)

<hr style="border: none; height: 5px; background-color: gray;">

### Calendar module


In [ ]:
# Weekday name for today
calendar.day_name[now.weekday()]

In [ ]:
# Check for leap year
calendar.isleap(2026)

In [ ]:
# Check for leap year
calendar.isleap(now.year)

In [ ]:
# number of days in current month
calendar.monthrange(now.year, now.month)[1]

In [ ]:
# list of weeks in a month (matrix format)
calendar.monthcalendar(2026, 4)

In [ ]:
# full month as text
print(calendar.month(2026, 3))

<hr style="border: none; height: 5px; background-color: gray;">

### Time module

In [ ]:
# Current UNIX time (seconds since 1970-01-01 UTC)
unix_timestamp = time.time()
unix_timestamp

In [ ]:
# Timing code: measure elapsed time
start = time.time()
time.sleep(1)  # Sleep for 1 second
elapsed = time.time() - start
elapsed

In [ ]:
# CLOCK_REALTIME: Uses system clock not monotonic time 
# not monotonic (can jump forward/backward)
# adjustable by the system (e.g. NTP or manual changes)
# resolution ≈ 1 microsecond

time.get_clock_info("time")

<hr style="border: none; height: 5px; background-color: gray;">

#### **Parsing** strings to datetime objects with `datetime` and `parser`

In [ ]:
# ISO-formatted string (works with datetime.fromisoformat)
dt_iso = datetime.fromisoformat('2026-03-30T15:00')
print(dt_iso)

In [ ]:
dt = datetime.strptime("30-03-2026 15:45", "%d-%m-%Y %H:%M")
print(dt)

In [ ]:
unix_timestamp = time.time()  

# Convert Unix timestamp to UTC (timezone-aware)
dt_utc = datetime.fromtimestamp(unix_timestamp, UTC)

# Convert Unix timestamp to local time (here: Europe/Zurich)
dt_local = datetime.fromtimestamp(unix_timestamp, ZoneInfo("Europe/Zurich"))

print("UTC:   ", dt_utc)     
print("Local: ", dt_local)  

In [ ]:
# Non-ISO string: will FAIL with datetime, but works with dateutil
try:
    datetime.fromisoformat('4th of July, 2015')  # Raises ValueError
except Exception as e:
    print(f"datetime.fromisoformat fails: {e}")

In [ ]:
# Flexible natural-language parsing with dateutil (works with many formats)
dt = parser.parse('4th of July, 2015 at 14:30')
print(dt)

In [ ]:
# Robust to different string formats
dt = parser.parse('2026/03/30 15:00')
print(dt)

In [ ]:
# dateutil parser can parse UTC offsets from strings
dt_one_hour_offset = parser.parse('2026-03-30T09:00:00+01:00')  # offset aware datetime
print(dt_one_hour_offset)

<hr style="border: none; height: 5px; background-color: gray;">

#### Formatting with `strftime()`

In [ ]:
# Format as custom string
dt = datetime(2026, 3, 30, 15, 0)
dt.strftime('%A, %d. %B %Y %H:%M')

In [ ]:
# Example: format output for filenames
dt.strftime('report_%Y%m%d_%H%M.txt')

##### Quick reference: common `strftime` codes

- `%Y`: year (2026)
- `%m`: month (03)
- `%d`: day (30)
- `%A`: weekday name (Saturday)
- `%B`: month name (March)
- `%H`: hour (15)
- `%M`: minute (00)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## NumPy time types

- NumPy has `datetime64` (points in time) and `timedelta64` (durations).
- All Pandas time objects are built on top of these types for vectorization and speed.
- Arrays and vector operations (differences, ranges) are efficient and fast.

In [ ]:
# datetime64: point in time (vectorized)
np.datetime64('2026-01-01 10:00')

In [ ]:
# timedelta64: durations
np.timedelta64(2, 'D')

In [ ]:
# Arrays of datetimes (fast, memory efficient)
dates = np.arange('2026-01', '2026-04', dtype='datetime64')
dates

In [ ]:
# Arrays of datetimes (fast, memory efficient)
dates = np.arange('2026-01', '2026-03', dtype='datetime64[D]')
dates[:8]  # first 5 days in January 2026

In [ ]:
# Operations: difference
np.datetime64('2026-02-01') - np.datetime64('2026-01-29')

In [ ]:
# Comparison: Python vs. NumPy vs. Pandas time objects

py_dt = datetime(2026, 1, 1, 10, 0)
np_dt = np.datetime64('2026-01-01 10:00')
pd_dt = pd.Timestamp('2026-01-01 10:00')

type(py_dt), type(np_dt), type(pd_dt)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Pandas time objects

- `pd.Timestamp`: high-level point in time (like Python's `datetime` but richer, time zone aware)
- `pd.Timedelta`: time difference (duration)
- `pd.Period`: calendar span (month, quarter, year, etc.)
- `pd.DatetimeIndex`, `TimedeltaIndex`, `PeriodIndex`: arrays of time for indexing and time series analysis

Pandas builds on NumPy's time types, but adds features like frequency, time zones, and indexing.

#### Timestamp

In [ ]:
# Timestamp: Exact point in time (high-level, extended datetime)
pd.Timestamp("2026-01-01 10:00")

In [ ]:
# Time-zone aware
pd.Timestamp("2026-01-01 10:00", tz="Europe/Zurich")

#### Timedelta

In [ ]:
 # Timedelta: A duration
pd.Timedelta(2, 'D')

#### Period

In [ ]:
# Period: Calendar interval
pd.Period("2026-01", freq="M")

<hr style="border: none; height: 5px; background-color: gray;">

### Pandas Time **Index** Types

- **Purpose:** Choose the appropriate index for your use-case:
    - `DatetimeIndex`: exact timestamps
    - `PeriodIndex`: regular periods (months, quarters…)
    - `TimedeltaIndex`: durations (intervals between events)

#### `DatetimeIndex`

In [ ]:
# Create a DatetimeIndex from start to end date with daily frequency (inclusive)
rng = pd.date_range(start="2026-01-01", end="2026-01-10", freq="D")
rng

In [ ]:
# Create a DatetimeIndex with 5 daily timestamps starting from 2026-01-01
rng = pd.date_range("2026-01-01", periods=5, freq="D")
print(type(rng[0]))
rng

In [ ]:
# Create a timezone-aware DatetimeIndex with 5 daily periods starting from 2026-01-01
rng = pd.date_range("2026-01-01", periods=5, freq="D", tz="Europe/Zurich")
rng

In [ ]:
# Business days
pd.date_range('2026-04-01', periods=5, freq='B')

<hr style="border: none; height: 5px; background-color: gray;">

#### PeriodIndex

In [ ]:
# PeriodIndex: array of periods (e.g. months)
pd.period_range("2026-01", periods=5, freq="M")

In [ ]:
# PeriodIndex: array of periods with start -> stop
pd.period_range('2026-01', '2026-12', freq='M')

In [ ]:
# PeriodIndex
print(pd.period_range('2026-01', periods=5, freq='min'))

<hr style="border: none; height: 5px; background-color: gray;">

#### `TimedeltaIndex`

In [ ]:
# TimedeltaIndex: array of durations
pd.to_timedelta(["1 day", "2 days", "5 hours"])

In [ ]:
# Timedelta range (every 12 hours)
print(pd.timedelta_range(start='0 days', periods=5, freq='12h'))

<hr style="border: none; height: 5px; background-color: gray;">

### Example: Time-Based Data in Pandas: DatetimeIndex, Periods, and Timedeltas

In [ ]:
# --- Create different time representations ---
dt_index = pd.date_range("2026-01-01", periods=10, freq="D", name="date")
period_index = pd.period_range("2026-01", periods=10, freq="D")
timedelta_index = pd.to_timedelta(np.arange(10), unit="D")

# --- Build DataFrame with named DatetimeIndex ---
df = pd.DataFrame(
    {
        "period": period_index,
        "delta": timedelta_index,
        "value": np.random.randn(10),
    },
    index=dt_index,
)
df

In [ ]:
# Add datetime column
df["timestamp"] = pd.date_range("2026-01-01 08:00", periods=10, freq="h")

In [ ]:
df.info()

<hr style="border: none; height: 5px; background-color: gray;">

#### The `.dt` Accessor for pandas Series (and DataFrame columns)


In [ ]:
df = pd.DataFrame({
    'timestamp': pd.date_range('2024-01-01', periods=6, freq='D'),
    'value': [10, 20, 30, 40, 50, 60]
})

df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['day'] = df['timestamp'].dt.day
df['dayofweek'] = df['timestamp'].dt.dayofweek

df

<hr style="border: none; height: 5px; background-color: gray;">

#### Influence of timezone when **converting** timezone-aware `Timestamps` to `Periods`

In [ ]:
# Create a Series of timezone-aware timestamps late in the day
ts = pd.Series(pd.date_range('2024-01-01 23:00:00', periods=4, freq='1h', tz='Europe/Zurich'))
print("Original timestamp Series with timezone:")
print(ts)

In [ ]:
# Convert the timestamps to daily periods (the local day)
periods_local = ts.dt.to_period('D')
print("Series as Periods (local time):")
print(periods_local)

In [ ]:
# Convert the timestamps to UTC for comparison
ts_utc = ts.dt.tz_convert('UTC')
print("Timestamps converted to UTC:")
print(ts_utc)

In [ ]:
# Convert the UTC timestamps to daily periods (the UTC day)
periods_utc = ts_utc.dt.to_period('D')
print("Series as Periods (UTC):")
print(periods_utc)

<hr style="border: none; height: 5px; background-color: gray;">

#### Example: Convert a `PeriodIndex` to a timezone-aware `DatetimeIndex` at the period end

In [ ]:
# Create a PeriodIndex (monthly frequency)
period_idx = pd.period_range('2024-01', periods=4, freq='M')
print(period_idx)

In [ ]:
# Convert PeriodIndex to DatetimeIndex (at the end of period)
# The result is timezone-naive --> localize to desired timezone
ts_idx = period_idx.to_timestamp(how='end').tz_localize('Europe/Berlin')
print(ts_idx)

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Date & Time Operations in Pandas


#### Datetime operations in Pandas

In [ ]:
# A range of days starting from day_date
day_date = pd.Timestamp('2024-04-01')
day_range = day_date + pd.to_timedelta(np.arange(12), 'D')
day_range

In [ ]:
# <: Little-endian byte order (not important for timestamps in practice)
# M8: NumPy's code for "datetime64" (M for "datetime", 8 bytes, 64 bit)
# [us]: Time unit is microseconds (us = microseconds)

print(type(day_range))
print(day_range.dtype)
day_range.dtype

In [ ]:
# Adding hours/minutes with pd.to_timedelta
base_time = pd.Timestamp('2024-04-01 08:00')
base_time + pd.to_timedelta([0, 1, 2, 3], 'h')

In [ ]:
# Subtracting dates to get differences (Timedelta)
start = pd.Timestamp('2024-04-01')
end = pd.Timestamp('2024-04-20')
end - start

In [ ]:
# Formatting dates with .strftime
ts = pd.Timestamp('2024-04-01 15:30')
ts.strftime('%Y/%m/%d %H:%M')

In [ ]:
# Shifting dates (offset vs. timedelta)
ts = pd.Timestamp('2024-01-31')
print("Original date:               ", ts)

# Using DateOffset (calendar-aware, handles month length correctly)
print("One month later (DateOffset):", ts + pd.DateOffset(months=1))  # 2024-02-29 (leap year!)

# Using Timedelta (fixed, 30 days)
print("30 days later (Timedelta):   ", ts + pd.Timedelta(days=30))    # 2024-03-01 (not end of Feb)

In [ ]:
# Rounding and normalizing timestamps
ts = pd.Timestamp('2024-04-01 15:44:25')
print("Original               :", ts)
print("Rounded to hour        :", ts.round('h'))
print("Normalized to midnight :", ts.normalize())

Use `.normalize()` to strip the time part from timestamps when you want to:
- Compare only dates (ignore hours/minutes/seconds)
- Use `.isin()` with date-based ranges like `pd.bdate_range()`

<hr style="border: none; height: 5px; background-color: gray;">

### Dateframe: Time-based Indexing

In [ ]:
# DatetimeIndex: Minute-frequency is 'min' (not 'T') in pandas≥2.2/3.0
dates = pd.date_range('2026-03-30', periods=5, freq='min')
df = pd.DataFrame({'value': np.arange(5)}, index=dates)
print(df.index)
df

In [ ]:
#### PeriodIndex: Month-frequency
periods = pd.period_range('2026-01', '2026-05', freq='M')
df = pd.DataFrame({'value': np.arange(len(periods))}, index=periods)
print(df.index)
df

In [ ]:
#### TimedeltaIndex
td_index = pd.timedelta_range(start='0 days', periods=6, freq='12h')
df_timedelta = pd.DataFrame({'value': np.arange(len(td_index))}, index=td_index)
print(df_timedelta.index)
df_timedelta

<hr style="border: none; height: 5px; background-color: gray;">

### Slicing by Time

In [ ]:
idx = pd.date_range('2026-04-01', '2026-04-30 23:59', freq='min')
df = pd.DataFrame({'value': np.random.randn(len(idx))}, index=idx)

In [ ]:
# Select specific days
df['2026-04-01':'2026-04-03'].head()

In [ ]:
# Select every 15th minute
df.iloc[::15].head()

<hr style="border: none; height: 5px; background-color: gray;">

#### boolean indexing

In [ ]:
# Select business days only
df_bdays = df[df.index.normalize().isin(pd.bdate_range('2026-04-01', '2026-04-30'))]

print("Unique days total:   ", df.index.normalize().nunique())
print("Unique business days:", df_bdays.index.normalize().nunique())

In [ ]:
# Restrict to 09:00–17:00 each day
selection = (df_bdays.index.time >= pd.to_datetime('09:00').time()) & (df_bdays.index.time <= pd.to_datetime('17:00').time())
df_selected = df_bdays[selection]

before_9 = (df_selected.index.time < pd.to_datetime('09:00').time()).sum()
after_17 = (df_selected.index.time > pd.to_datetime('17:00').time()).sum()
print("Count of times before 09:00:", before_9)
print("Count of times after 17:00:", after_17)

<hr style="border: none; height: 5px; background-color: gray;">

### Parsing with `pd.to_datetime()`

In [ ]:
dates = ['2026-03-30', 'March 31, 2026', '2026/04/01 08:10']
pd.to_datetime(dates, format="mixed")

#### Load data from CSV and convert into DataFrame with DatetimeIndex

In [ ]:
# Load air quality data from CSV
df_air_quality = pd.read_csv('../data/csv/air_quality_no2.csv')
df_air_quality.head()

In [ ]:
# Set the 'datetime' column as the index
df_air_quality = df_air_quality.set_index('datetime')
print(type(df_air_quality.index))
print(df_air_quality.index.dtype)
df_air_quality.head()

In [ ]:
# Parse the index to a DateTimeIndex (make sure it's not just a string index)
df_air_quality.index = pd.to_datetime(df_air_quality.index)
print(type(df_air_quality.index))
print(df_air_quality.index.dtype)
df_air_quality.head()


In [ ]:
# Test: Select all rows from a specific day using the DateTimeIndex
df_air_quality.loc['2019-05-08'].head()

#### Load CSV data and directly create a DateTimeIndex in one step

In [ ]:
df_air_quality = pd.read_csv(
    '../data/csv/air_quality_no2.csv',
    parse_dates=["datetime"],
    index_col="datetime"
)

df_air_quality.info()

<hr style="border: none; height: 10px; background-color: LightBlue;">

## Air Quality Analysis: Resampling and Rolling

#### Load the air quality data

In [ ]:
df_air_quality = pd.read_csv(
    '../data/csv/air_quality_no2.csv',
    parse_dates=["datetime"],
    index_col="datetime"
)


In [ ]:
# Inspect first rows and plot raw data (Paris station)
df_air_quality.head()

In [ ]:
# Data info
df_air_quality.info()

In [ ]:
# Quick plot
df_air_quality['station_paris'].plot(figsize=(12, 4), title="NO₂ raw readings (Paris)");
plt.tight_layout()


<hr style="border: none; height: 5px; background-color: gray;">

### Resampling: compute 3-hour mean for Paris station

In [ ]:
df_air_quality[['station_paris']].head()

In [ ]:
df_paris_3h = df_air_quality[['station_paris']].resample('3h').mean()
df_paris_3h.head()

In [ ]:
# Resample tp 3-hour averages
df_air_quality[['station_paris']].resample('3h').mean().head()

<hr style="border: none; height: 5px; background-color: gray;">

### Plot original vs daily mean

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

df_air_quality['station_paris'].plot(ax=ax, alpha=0.5, label='original')

df_paris_day = df_air_quality['station_paris'].resample('1D').mean()
df_paris_day.plot(ax=ax, linewidth=2, label='daily mean')

ax.set_title("Resampling: Daily Mean (Paris)")
plt.legend();

<hr style="border: none; height: 5px; background-color: gray;">

### Resampling: daily maximum (Paris station)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

df_air_quality['station_paris'].plot(ax=ax, alpha=0.5, label='original')

df_paris_day.plot(ax=ax, linewidth=2, label='daily mean')

df_paris_max = df_air_quality['station_paris'].resample('1D').max()
df_paris_max.plot(ax=ax, linewidth=2, label='daily max')

ax.set_title("Resampling: Daily Mean vs Max (Paris)")
plt.legend();

<hr style="border: none; height: 5px; background-color: gray;">

### Resampling: weekly mean for all stations

In [ ]:
df_weekly = df_air_quality.resample('W').mean()  
df_weekly.plot(figsize=(12, 4), marker='o', title="Weekly Means (All Stations)")

<hr style="border: none; height: 5px; background-color: gray;">

### Rolling window: daily rolling mean (Paris station)

In [ ]:
df_air_quality[['station_paris']].head(10)

In [ ]:
df_air_quality[['station_paris']].rolling('6h').mean().head(10)

In [ ]:
# Rolling: daily Mean (Paris)
df_rolling_6h = df_air_quality['station_paris'].rolling('24h').mean()

ax = df_air_quality['station_paris'].plot(figsize=(12, 4), alpha=0.5, label='Original')
df_rolling_6h.plot(ax=ax, linewidth=2, label='Daily Rolling Mean', title="Rolling: daily Mean (Paris)")
plt.legend()

<hr style="border: none; height: 5px; background-color: gray;">

### Rolling: 6-hour rolling mean for all stations

In [ ]:
df_rolling_mean_all = df_air_quality.rolling('6h').mean()
df_rolling_mean_all.plot(figsize=(12, 4), title="Rolling: 6-hour Mean (All Stations)", alpha=0.85)

<hr style="border: none; height: 5px; background-color: gray;">

### Comparison: Daily Rolling Mean vs. Daily Resampled Mean (Paris NO₂)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

# Rolling daily mean
df_rolling_day = df_air_quality['station_paris'].rolling('1D').mean()
df_rolling_day.plot(ax=ax, linewidth=2, label='Daily Rolling Mean')

# Daily resampled mean
df_resample_day = df_air_quality['station_paris'].resample('1D').mean()
df_resample_day.plot(ax=ax, linewidth=2, label='Daily Mean (Resampled)')

ax.set_title("Comparison: Daily Rolling Mean vs. Daily Resampled Mean (Paris NO₂)")
plt.legend();

<hr style="border: none; height: 5px; background-color: gray;">

### Timeseries Example from the script

In [ ]:
import matplotlib.pyplot as plt

df_air_quality = pd.read_csv('../data/csv/air_quality_no2.csv', parse_dates=["datetime"])

start, end = "2019-05-07", "2019-05-15"

plt.figure(figsize=(8, 4))
colors = {"station_london": "#2062AB", "station_paris": "#E67E22"}
labels = {"station_london": "London", "station_paris": "Paris"}

for col in ["station_london", "station_paris"]:
    serie = df_air_quality[["datetime", col]].dropna()
    serie = serie[(serie["datetime"] >= start) & (serie["datetime"] < end)]
    plt.plot(
        serie["datetime"],
        serie[col],
        marker="o",
        linewidth=2,
        color=colors[col],
        label=labels[col],
        alpha=0.85,
        markersize=4,
    )

plt.title("NO₂ Concentration (Example Time Series)")
plt.xlabel("Time")
plt.xticks(rotation=20) 
plt.ylabel("NO₂ [μg/m³]")
plt.grid(axis="both", alpha=0.3)
plt.tight_layout()
plt.legend()
plt.show()